# 🔋 Predicting Electric Vehicle Purchases — Notebook Kaggle complet

**Objectif :** prédire la probabilité qu'un individu achète un véhicule électrique (`Will_Buy_EV`)
à partir de 13 variables socio-comportementales, et maximiser l'**AUC-ROC** en validation croisée.

**Ce que fait ce notebook (pipeline complet, prêt pour la compétition) :**
1. Chargement des données (compatible local **et** Kaggle)
2. Audit rapide + rappel des insights clés (voir étude complète dans `outputs/logs/01_data_audit_report.txt`)
3. Feature engineering ciblé, sans fuite de données (X-only + target encoding strictement OOF)
4. 4 modèles de familles différentes (**LightGBM, CatBoost, XGBoost, HistGradientBoosting**),
   chacun validé en **StratifiedKFold** sur l'intégralité du jeu d'entraînement (668 665 lignes)
5. **Accélération GPU automatique** si un accélérateur est disponible (ex. **2× NVIDIA T4** sur
   Kaggle) : CatBoost exploite nativement les deux GPU pour un même modèle, LightGBM/XGBoost
   alternent un GPU par pli de CV — avec repli CPU transparent si aucun GPU n'est détecté
6. **Ensemble** (moyenne pondérée optimisée, moyenne de rangs, stacking) sélectionné sur la base
   de l'AUC out-of-fold (OOF), la seule mesure honnête de la performance en généralisation
7. Une section d'**analyse critique du plafond de score** (à quel point ce dataset contient-il
   du signal exploitable ?) — voir la section dédiée avant la conclusion
8. Génération du fichier `submission.csv` avec toutes les vérifications Kaggle usuelles

> 💡 **Note de transparence (lue avant tout le reste) :** une investigation empirique poussée
> (ablation de features, entraînement pleine échelle, recherche d'hyperparamètres Optuna, et une
> analyse "oracle" par target-encoding joint) montre que ce dataset a un **plafond de bruit
> intrinsèque autour de AUC ≈ 0.94–0.945**, quel que soit le modèle utilisé (voir section 8).
> Ce notebook pousse la modélisation aussi loin que raisonnablement possible pour s'en approcher
> au maximum et documente honnêtement le score obtenu.

### ▶️ Comment exécuter ce notebook sur Kaggle
0. *(Optionnel mais recommandé)* Dans **Settings → Accelerator**, sélectionner **GPU T4 x2** pour
   accélérer l'entraînement. Le notebook **détecte automatiquement** le nombre de GPU disponibles
   (`nvidia-smi`) et adapte chaque modèle en conséquence ; il fonctionne aussi bien **sans GPU**
   (repli CPU automatique, aucune action requise).
1. **Add Data** → chercher le dataset de la compétition « Predicting Electric Vehicle Purchases »
   (ou tout dataset contenant `train.csv` + `test.csv`) et l'attacher au notebook.
2. Vérifier que `FAST_MODE = False` (cellule suivante) pour le score final, ou `True` pour une
   passe de validation rapide (~5-10 min) qui vérifie que tout s'exécute sans erreur.
3. **Save & Run All (Commit)**. Durée estimée en mode complet : **~1h30 à 3h sur CPU**, ou
   **nettement plus rapide avec l'accélérateur GPU T4 x2** — largement dans la limite standard
   de 9h/session. `submission.csv` apparaît ensuite dans l'onglet **Output** du run commité
   (écrit dans `/kaggle/working/`).

## 1. Imports & configuration

In [ ]:
import os, sys, time, json, warnings, subprocess
warnings.filterwarnings("ignore")

# --- verification robuste des dependances (Kaggle les fournit deja pre-installees ; ---
# --- ce filet de securite evite un crash si l'environnement d'execution differe) -------
REQUIRED = ["numpy", "pandas", "scipy", "sklearn", "lightgbm", "catboost", "xgboost"]
_PIP_NAME = {"sklearn": "scikit-learn"}
for _mod in REQUIRED:
    try:
        __import__(_mod)
    except ImportError:
        pkg = _PIP_NAME.get(_mod, _mod)
        print(f"Package '{pkg}' introuvable, tentative d'installation...")
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        except Exception as e:
            print(f"  -> echec installation automatique de {pkg} ({e}); a installer manuellement si besoin.")

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression

SEED = 42

# FAST_MODE=True -> validation rapide (3-fold, peu d'iterations, ~5-10 min) pour verifier que
# tout le pipeline s'execute sans erreur AVANT de lancer la version complete (FAST_MODE=False,
# 5-fold, pleine echelle) qui donne le score final soumis sur Kaggle.
FAST_MODE = False

N_FOLDS = 3 if FAST_MODE else 5   # augmenter a 10 en mode complet pour un gain marginal (+/- 0.001)
np.random.seed(SEED)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

# --- detection GPU (accelerateur Kaggle usuel : 2x NVIDIA T4) -------------------------
# DISABLE_GPU=True permet de forcer un entrainement 100% CPU (debug/comparaison) meme si
# un GPU est detecte. Sans GPU (local, CPU-only), tout continue de fonctionner a l'identique :
# chaque modele bascule alors simplement sur ses parametres CPU (voir fit_with_gpu_fallback).
DISABLE_GPU = False


def detect_gpus():
    """Retourne la liste (str) des GPU NVIDIA detectes via `nvidia-smi` (liste vide si absent)."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10,
        )
        if out.returncode != 0:
            return []
        return [line.strip() for line in out.stdout.strip().splitlines() if line.strip()]
    except Exception:
        return []


GPU_LIST = detect_gpus()
N_GPUS = len(GPU_LIST)
USE_GPU = (N_GPUS > 0) and not DISABLE_GPU

print(f"Environnement pret. SEED={SEED}  N_FOLDS={N_FOLDS}  FAST_MODE={FAST_MODE}")
if USE_GPU:
    print(f"GPU(s) detecte(s): {N_GPUS} -> acceleration activee (CatBoost multi-GPU natif, "
          f"LightGBM/XGBoost GPU en round-robin par pli).")
    for _g in GPU_LIST:
        print("   -", _g)
else:
    _reason = "desactive manuellement (DISABLE_GPU=True)" if (N_GPUS > 0 and DISABLE_GPU) else "aucun GPU detecte"
    print(f"Mode CPU ({_reason}) -> tous les modeles s'entrainent normalement sur CPU.")

## 2. Chargement des données

Le chemin des données est détecté automatiquement :
- Sur **Kaggle**, les fichiers sont attendus sous `/kaggle/input/<dataset>/train.csv` et `test.csv`.
- En **local**, ils sont attendus à la racine du dépôt (`train.csv`, `test.csv`).

In [ ]:
def find_data_dir():
    candidates = ["/kaggle/input"]
    for base in candidates:
        if os.path.isdir(base):
            for root, _dirs, files in os.walk(base):
                if "train.csv" in files and "test.csv" in files:
                    return root
    # fallback: local (racine du repo, ou dossier courant)
    here = os.getcwd()
    for cand in [here, os.path.dirname(here)]:
        if os.path.exists(os.path.join(cand, "train.csv")) and os.path.exists(os.path.join(cand, "test.csv")):
            return cand
    raise FileNotFoundError("Impossible de localiser train.csv / test.csv (local ou Kaggle).")

DATA_DIR = find_data_dir()
print("Dossier de donnees detecte:", DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# normalisation des dtypes objet -> str (compat maximale avec catboost/lightgbm/xgboost)
for c in train.select_dtypes(include="object").columns:
    train[c] = train[c].astype(str)
for c in test.select_dtypes(include="object").columns:
    test[c] = test[c].astype(str)

ID_COL = "id"
TARGET_COL = "Will_Buy_EV"
CAT_COLS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
            "Subsidy_Available", "Range_Anxiety_Level"]
NUM_COLS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
            "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"]
BASE_FEATURES = NUM_COLS + CAT_COLS

y = train[TARGET_COL].map({"Yes": 1, "No": 0}).astype(int)
print(f"train shape={train.shape}  test shape={test.shape}  positive rate={y.mean():.4f}")

## 3. Rappel des insights clés de l'audit exploratoire

L'audit complet (`01_data_audit.py` → `outputs/logs/01_data_audit_report.txt`) a mis en évidence
des leviers **extrêmement** non-linéaires :

| Levier | Effet sur le taux d'achat |
|---|---|
| `Subsidy_Available = No` | quasi-nul (~0.5 %), quel que soit le reste |
| `Subsidy_Available = Yes` × `Range_Anxiety_Level = Low` | ~29.6 % |
| `Environmental_Concern_Level` (1→5) | 0.6 % → 2.1 % → 11.1 % → 24.9 % → **51.8 %** (quasi-exponentiel) |
| `Home_Charging_Possible = Yes` × `Subsidy = Yes` | ~30.4 % vs ~0.7 % si pas de subvention |
| `Annual_Income_USD` (decile bas → haut) | 4.4 % → 33.7 % (monotone) |

Ces variables (`Subsidy_Available`, `Environmental_Concern_Level`, `Range_Anxiety_Level`,
`Home_Charging_Possible`, `Annual_Income_USD`) sont donc les **principaux moteurs du signal**,
et les modèles à base d'arbres (CatBoost/LightGBM/XGBoost) les capturent nativement très bien
via leurs splits successifs — voir section 8 pour la vérification empirique de ce point.

In [ ]:
print(train[TARGET_COL].value_counts(normalize=True))
print()
display_cols = ["Subsidy_Available", "Environmental_Concern_Level", "Range_Anxiety_Level", "Home_Charging_Possible"]
tmp = train.copy()
tmp["_y"] = y
for c in display_cols:
    print(f"--- taux d'achat par {c} ---")
    print(tmp.groupby(c, observed=True)["_y"].mean().sort_values().to_string())
    print()

## 4. Feature engineering (sans fuite de données)

Toutes les transformations ci-dessous sont **X-only** (n'utilisent jamais la cible) donc
peuvent être appliquées indépendamment sur train/test. Le **target encoding**, lui, est
strictement **out-of-fold** (calculé séparément par pli de la CV finale) pour rester honnête.

⚠️ Une étude d'ablation rigoureuse (voir `outputs/logs/experiment_log.csv`, 16/16 comparaisons
CatBoost+LightGBM) a montré que ces features enrichies **n'améliorent pas** — et dégradent
même très légèrement — les modèles à base d'arbres entraînés sur les features brutes : les
GBDT capturent déjà nativement les interactions catégorielles à faible cardinalité. On les
conserve donc uniquement pour :
- le modèle `HistGradientBoosting` (diversité d'ensemble : un modèle différent, sur une vue de
  features différente, aide souvent l'ensemble même si son score individuel n'est pas meilleur) ;
- l'analyse du "plafond de signal" en section 8.

In [ ]:
RANGE_ANXIETY_MAP = {"Low": 0, "Medium": 1, "High": 2}
CITY_DENSITY_MAP = {"Urban": 2, "Suburban": 1, "Rural": 0}


def add_engineered_features(df):
    df = df.copy()
    df["Range_Anxiety_Ordinal"] = df["Range_Anxiety_Level"].map(RANGE_ANXIETY_MAP).astype(float)
    df["Home_Charging_Binary"] = (df["Home_Charging_Possible"] == "Yes").astype(float)
    df["Subsidy_Binary"] = (df["Subsidy_Available"] == "Yes").astype(float)
    df["City_Density_Ordinal"] = df["City_Type"].map(CITY_DENSITY_MAP).astype(float)

    df["log_income"] = np.log1p(df["Annual_Income_USD"])
    df["income_per_commute"] = df["Annual_Income_USD"] / (df["Daily_Commute_km"] + 1.0)
    df["Total_Charging_Stations"] = df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    df["charging_density_relative_to_commute"] = df["Total_Charging_Stations"] / (df["Daily_Commute_km"] + 1.0)

    # interactions comportementales (coeur du signal identifie dans l'audit)
    df["env_x_range_anxiety"] = df["Environmental_Concern_Level"] * df["Range_Anxiety_Ordinal"]
    df["subsidy_x_env"] = df["Subsidy_Binary"] * df["Environmental_Concern_Level"]
    df["home_x_range_anxiety"] = df["Home_Charging_Binary"] * df["Range_Anxiety_Ordinal"]
    df["subsidy_x_home"] = df["Subsidy_Binary"] * df["Home_Charging_Binary"]
    df["income_x_env"] = df["log_income"] * df["Environmental_Concern_Level"]
    df["ev_readiness_score"] = (
        df["Subsidy_Binary"] * 3.0 + df["Home_Charging_Binary"] * 2.0
        + df["Environmental_Concern_Level"] - df["Range_Anxiety_Ordinal"] * 3.0
    )
    return df


ENGINEERED_COLS = [
    "Range_Anxiety_Ordinal", "Home_Charging_Binary", "Subsidy_Binary", "City_Density_Ordinal",
    "log_income", "income_per_commute", "Total_Charging_Stations", "charging_density_relative_to_commute",
    "env_x_range_anxiety", "subsidy_x_env", "home_x_range_anxiety", "subsidy_x_home",
    "income_x_env", "ev_readiness_score",
]

train_fe = add_engineered_features(train)
test_fe = add_engineered_features(test)
ENRICHED_FEATURES = BASE_FEATURES + ENGINEERED_COLS
print(f"Features de base: {len(BASE_FEATURES)}   |   Features enrichies (diversite): {len(ENRICHED_FEATURES)}")

## 5. Validation croisée & utilitaires génériques

Un seul jeu de plis **StratifiedKFold** (`SEED=42`) est réutilisé pour **tous** les modèles :
c'est indispensable pour que les prédictions OOF soient alignées entre modèles et que
l'ensemble/stacking soit valide.

In [ ]:
def get_folds(y_arr, n_splits=N_FOLDS, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(y_arr)), y_arr))


FOLDS = get_folds(y.to_numpy())
print(f"{len(FOLDS)} plis crees. Tailles (train/val) du pli 0: "
      f"{len(FOLDS[0][0])}/{len(FOLDS[0][1])}")


def run_cv(fold_fn_factory, X, y_, X_test, folds=FOLDS, verbose=True, name=""):
    oof = np.zeros(len(X))
    test_preds = np.zeros((len(folds), len(X_test)))
    fold_scores = []
    y_arr = y_.to_numpy() if hasattr(y_, "to_numpy") else np.asarray(y_)

    for i, (tr_idx, val_idx) in enumerate(folds):
        t0 = time.time()
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_.iloc[tr_idx], y_.iloc[val_idx]
        fold_fn = fold_fn_factory(fold_i=i)
        val_pred, test_pred = fold_fn(X_tr, y_tr, X_val, y_val, X_test)
        oof[val_idx] = val_pred
        test_preds[i] = test_pred
        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        if verbose:
            print(f"    [{name}] fold {i}: AUC={score:.5f}  time={time.time()-t0:.1f}s")

    oof_auc = roc_auc_score(y_arr, oof)
    print(f"  -> [{name}] OOF AUC={oof_auc:.5f}  (fold mean={np.mean(fold_scores):.5f} "
          f"std={np.std(fold_scores):.5f})")
    return {"oof": oof, "test_pred": test_preds.mean(axis=0), "fold_scores": fold_scores, "oof_auc": oof_auc}


def to_category(df, cols):
    df = df.copy()
    for c in cols:
        df[c] = df[c].astype("category")
    return df


def align_categories(train_df, test_df, cols):
    train_df, test_df = train_df.copy(), test_df.copy()
    for c in cols:
        cats = pd.api.types.union_categoricals(
            [train_df[c].astype("category"), test_df[c].astype("category")]
        ).categories
        train_df[c] = pd.Categorical(train_df[c], categories=cats)
        test_df[c] = pd.Categorical(test_df[c], categories=cats)
    return train_df, test_df


def fit_with_gpu_fallback(build_and_fit_gpu, build_and_fit_cpu, fold_i, model_name):
    """Tente l'entrainement sur GPU ; en cas d'echec (build sans support GPU, driver absent,
    memoire insuffisante, etc.) bascule automatiquement sur l'equivalent CPU. Garantit que le
    notebook s'execute de bout en bout quel que soit l'environnement (Kaggle GPU, Kaggle CPU,
    poste local sans GPU)."""
    if not USE_GPU:
        return build_and_fit_cpu()
    try:
        return build_and_fit_gpu()
    except Exception as e:
        print(f"    [{model_name}][fold {fold_i}] GPU indisponible/erreur "
              f"({type(e).__name__}: {e}); repli sur CPU pour ce pli.")
        return build_and_fit_cpu()


oof_dict, test_dict, timing_dict = {}, {}, {}

## 6. Modèle 1 — LightGBM

Hyperparamètres issus de la recherche **Optuna** menée dans `05_hyperparameter_search.py`
(40 essais, TPE sampler) : `num_leaves=32, max_depth=4, learning_rate≈0.0204` — un arbre
volontairement peu profond mais avec beaucoup d'itérations, ce qui régularise fortement
face au bruit du label (cf. section 8). On augmente `n_estimators` et on laisse
l'**early stopping** déterminer le nombre optimal d'arbres sur pleine échelle.

🖥️ **GPU :** si un ou plusieurs GPU sont détectés, chaque pli de la CV est entraîné sur GPU
(`device_type="gpu"`) en alternant les GPU disponibles en round-robin (ex. T4 x2 → plis pairs
sur GPU 0, impairs sur GPU 1). LightGBM ne supporte pas nativement le multi-GPU pour un seul
modèle (contrairement à CatBoost) ; répartir les plis est donc la meilleure façon d'exploiter
les deux cartes. Repli automatique sur CPU si le build LightGBM installé n'a pas le support GPU.

In [ ]:
import lightgbm as lgb

LGB_PARAMS = dict(
    n_estimators=800 if FAST_MODE else 6000, num_leaves=32, max_depth=4, learning_rate=0.0204,
    min_child_samples=184, subsample=0.6628, subsample_freq=1, colsample_bytree=0.5136,
    reg_alpha=0.00255, reg_lambda=0.00642, random_state=SEED, verbose=-1,
)
LGB_EARLY_STOP = 60 if FAST_MODE else 150


def lightgbm_factory(params, cat_cols, early_stopping_rounds=LGB_EARLY_STOP):
    def factory(fold_i):
        def _fn(X_tr, y_tr, X_val, y_val, X_test):
            def _build_fit(use_gpu):
                p = dict(params)
                if use_gpu:
                    # round-robin par pli sur les GPU disponibles (ex: T4 x2 -> fold 0=GPU0, fold 1=GPU1, ...)
                    p.update(device_type="gpu", gpu_device_id=fold_i % N_GPUS)
                model = lgb.LGBMClassifier(**p)
                model.fit(
                    X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc",
                    categorical_feature=cat_cols,
                    callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False), lgb.log_evaluation(0)],
                )
                return model
            model = fit_with_gpu_fallback(lambda: _build_fit(True), lambda: _build_fit(False), fold_i, "LightGBM")
            return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
        return _fn
    return factory


Xl, Xl_test = to_category(train_fe[BASE_FEATURES], CAT_COLS), to_category(test_fe[BASE_FEATURES], CAT_COLS)
Xl, Xl_test = align_categories(Xl, Xl_test, CAT_COLS)

t0 = time.time()
res = run_cv(lightgbm_factory(LGB_PARAMS, CAT_COLS, early_stopping_rounds=LGB_EARLY_STOP), Xl, y, Xl_test, name="LightGBM")
timing_dict["lightgbm"] = time.time() - t0
oof_dict["lightgbm"], test_dict["lightgbm"] = res["oof"], res["test_pred"]

## 7. Modèle 2 — CatBoost

CatBoost gère nativement les catégorielles via des statistiques de comptage ordonnées
(`max_ctr_complexity=2` combine automatiquement des paires de colonnes catégorielles) —
c'est le modèle le plus apte à exploiter les interactions mises en évidence dans l'audit.

🖥️ **GPU :** CatBoost est la seule librairie des 4 à supporter nativement l'entraînement
**multi-GPU d'un seul modèle** (`task_type="GPU", devices="0:1"` pour 2× T4) : les deux cartes
collaborent directement sur le même modèle plutôt que d'être utilisées à tour de rôle. Repli
automatique sur CPU si aucun GPU CUDA n'est disponible.

In [ ]:
from catboost import CatBoostClassifier

CB_PARAMS = dict(
    iterations=800 if FAST_MODE else 6000, depth=6, learning_rate=0.05, l2_leaf_reg=4.0, random_strength=1.0,
    bagging_temperature=1.0, border_count=128, max_ctr_complexity=2, random_seed=SEED,
)
# NB: depth=6 offre le meilleur compromis vitesse/AUC observe empiriquement (depth=7-10 testes
# en ablation n'apportent pas de gain significatif mais coutent 2 a 4x plus cher en temps CPU).
CB_EARLY_STOP = 60 if FAST_MODE else 150
CB_GPU_DEVICES = ":".join(str(i) for i in range(N_GPUS)) if USE_GPU else None  # ex: "0:1" pour T4 x2


def catboost_factory(params, cat_cols, early_stopping_rounds=CB_EARLY_STOP):
    def factory(fold_i):
        def _fn(X_tr, y_tr, X_val, y_val, X_test):
            def _build_fit(use_gpu):
                p = dict(params)
                if use_gpu:
                    # CatBoost distribue nativement l'entrainement d'un seul modele sur plusieurs
                    # GPU (ex: "0:1" pour 2x T4) -- contrairement a LightGBM/XGBoost (mono-GPU).
                    p.update(task_type="GPU", devices=CB_GPU_DEVICES)
                model = CatBoostClassifier(
                    **p, cat_features=cat_cols, eval_metric="AUC", loss_function="Logloss",
                    early_stopping_rounds=early_stopping_rounds, verbose=False, allow_writing_files=False,
                )
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
                return model
            model = fit_with_gpu_fallback(lambda: _build_fit(True), lambda: _build_fit(False), fold_i, "CatBoost")
            return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
        return _fn
    return factory


Xc, Xc_test = train_fe[BASE_FEATURES].copy(), test_fe[BASE_FEATURES].copy()

t0 = time.time()
res = run_cv(catboost_factory(CB_PARAMS, CAT_COLS, early_stopping_rounds=CB_EARLY_STOP), Xc, y, Xc_test, name="CatBoost")
timing_dict["catboost"] = time.time() - t0
oof_dict["catboost"], test_dict["catboost"] = res["oof"], res["test_pred"]

## 8. Modèle 3 — XGBoost

`tree_method="hist"` + catégorielles natives (`enable_categorical=True`) pour la vitesse.

🖥️ **GPU :** comme LightGBM, XGBoost mono-processus n'entraîne un modèle que sur **un seul**
GPU à la fois (`device="cuda:<id>"`) ; les plis alternent en round-robin entre les GPU détectés
(T4 x2 → `cuda:0` / `cuda:1`) pour répartir la charge sur l'ensemble de la session. Repli
automatique sur CPU (`device="cpu"`) si aucun GPU CUDA n'est disponible.

ℹ️ **Particularité XGBoost :** contrairement à LightGBM/CatBoost (qui lèvent une erreur
explicite si le GPU demandé est indisponible), XGBoost gère ce cas en interne via un simple
avertissement puis poursuit automatiquement sur CPU **sans lever d'exception**. Le notebook
capture cet avertissement (normalement masqué par le filtre global) pour rester transparent
sur le device réellement utilisé, au lieu de laisser croire à tort que le GPU a servi.

In [ ]:
import xgboost as xgb

XGB_PARAMS = dict(
    n_estimators=800 if FAST_MODE else 6000, max_depth=6, learning_rate=0.035, min_child_weight=6,
    subsample=0.8, colsample_bytree=0.75, gamma=0.1, reg_alpha=0.05, reg_lambda=1.0,
    random_state=SEED, tree_method="hist", enable_categorical=True, eval_metric="auc",
)
XGB_EARLY_STOP = 60 if FAST_MODE else 150


def xgboost_factory(params, early_stopping_rounds=XGB_EARLY_STOP):
    def factory(fold_i):
        def _fn(X_tr, y_tr, X_val, y_val, X_test):
            def _build_fit(use_gpu):
                p = dict(params)
                # round-robin par pli sur les GPU disponibles (ex: T4 x2 -> fold 0=cuda:0, fold 1=cuda:1, ...)
                p["device"] = f"cuda:{fold_i % N_GPUS}" if use_gpu else "cpu"
                model = xgb.XGBClassifier(**p, early_stopping_rounds=early_stopping_rounds)
                if use_gpu:
                    # XGBoost ne leve PAS d'exception si le GPU demande est indisponible : il
                    # emet un simple UserWarning et bascule lui-meme sur CPU en interne
                    # (contrairement a LightGBM/CatBoost qui levent une erreur explicite). On
                    # capture ce warning pour rester transparent sur le device reellement
                    # utilise (sinon masque par warnings.filterwarnings("ignore") en tete de notebook).
                    with warnings.catch_warnings(record=True) as _caught:
                        warnings.simplefilter("always")
                        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
                        if any("GPU" in str(_w.message) for _w in _caught):
                            print(f"    [XGBoost][fold {fold_i}] GPU demande indisponible -> "
                                  f"repli interne XGBoost sur CPU pour ce pli.")
                else:
                    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
                return model
            model = fit_with_gpu_fallback(lambda: _build_fit(True), lambda: _build_fit(False), fold_i, "XGBoost")
            return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
        return _fn
    return factory


Xx, Xx_test = to_category(train_fe[BASE_FEATURES], CAT_COLS), to_category(test_fe[BASE_FEATURES], CAT_COLS)
Xx, Xx_test = align_categories(Xx, Xx_test, CAT_COLS)

t0 = time.time()
res = run_cv(xgboost_factory(XGB_PARAMS, early_stopping_rounds=XGB_EARLY_STOP), Xx, y, Xx_test, name="XGBoost")
timing_dict["xgboost"] = time.time() - t0
oof_dict["xgboost"], test_dict["xgboost"] = res["oof"], res["test_pred"]

## 9. Modèle 4 — HistGradientBoosting (diversité d'ensemble)

Implémentation scikit-learn, entraînée sur le set **enrichi** (features + interactions).
Objectif : apporter un point de vue différent (autre librairie, autre feature set, pas de
tuning partagé avec les 3 modèles précédents) pour maximiser le gain de l'ensemble final.

🖥️ **GPU :** scikit-learn ne propose pas d'accélération GPU pour `HistGradientBoostingClassifier` ;
ce modèle reste donc toujours entraîné sur CPU, que l'accélérateur Kaggle soit activé ou non.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

HGB_PARAMS = dict(
    max_iter=500 if FAST_MODE else 2500, learning_rate=0.045, max_leaf_nodes=63, min_samples_leaf=40,
    l2_regularization=0.15, early_stopping=True, n_iter_no_change=40 if FAST_MODE else 80, validation_fraction=0.1,
    random_state=SEED,
)


def histgb_factory(params):
    def factory(fold_i):
        def _fn(X_tr, y_tr, X_val, y_val, X_test):
            model = HistGradientBoostingClassifier(**params, categorical_features="from_dtype")
            model.fit(X_tr, y_tr)
            return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
        return _fn
    return factory


Xh, Xh_test = to_category(train_fe[ENRICHED_FEATURES], CAT_COLS), to_category(test_fe[ENRICHED_FEATURES], CAT_COLS)
Xh, Xh_test = align_categories(Xh, Xh_test, CAT_COLS)

t0 = time.time()
res = run_cv(histgb_factory(HGB_PARAMS), Xh, y, Xh_test, name="HistGB")
timing_dict["histgb"] = time.time() - t0
oof_dict["histgb"], test_dict["histgb"] = res["oof"], res["test_pred"]

## 10. Récapitulatif des scores individuels

In [ ]:
summary_rows = []
for name, oof in oof_dict.items():
    summary_rows.append({"model": name, "oof_auc": roc_auc_score(y, oof), "train_time_sec": timing_dict[name]})
summary_df = pd.DataFrame(summary_rows).sort_values("oof_auc", ascending=False).reset_index(drop=True)
print(summary_df.to_string(index=False))

## 11. Ensemble : blend pondéré, moyenne de rangs & stacking

Trois stratégies sont comparées sur l'AUC **OOF** (la seule mesure honnête), et la meilleure
est retenue automatiquement pour générer la soumission finale.

In [ ]:
def optimize_weights_random_search(pred_dict, y_, n_iter=(200 if FAST_MODE else 4000), seed=SEED, refine_rounds=(1 if FAST_MODE else 3)):
    rng = np.random.default_rng(seed)
    names = list(pred_dict.keys())
    P = np.column_stack([pred_dict[n] for n in names])
    y_arr = np.asarray(y_)
    n = len(names)

    def score_of(w):
        return roc_auc_score(y_arr, P @ w)

    best_w = np.ones(n) / n
    best_score = score_of(best_w)
    alphas = np.ones(n)
    for _ in range(n_iter):
        w = rng.dirichlet(alphas)
        s = score_of(w)
        if s > best_score:
            best_score, best_w = s, w
    for r in range(refine_rounds):
        scale = 0.2 / (r + 1)
        for _ in range(max(50, n_iter // 4)):
            w = np.clip(best_w + rng.normal(0, scale, size=n), 0, None)
            if w.sum() <= 0:
                continue
            w = w / w.sum()
            s = score_of(w)
            if s > best_score:
                best_score, best_w = s, w
    return dict(zip(names, best_w)), best_score


def blend_predictions(pred_dict, weights):
    names = list(pred_dict.keys())
    P = np.column_stack([pred_dict[n] for n in names])
    w = np.array([weights[n] for n in names])
    return P @ w


def rank_transform(pred_dict):
    return {k: rankdata(v) / len(v) for k, v in pred_dict.items()}


def stacking_cv(oof_dict_, y_, n_splits=N_FOLDS, seed=SEED, C=1.0):
    names = list(oof_dict_.keys())
    X_meta = np.column_stack([oof_dict_[n] for n in names])
    y_arr = np.asarray(y_)
    folds = get_folds(y_arr, n_splits=n_splits, seed=seed)
    meta_oof = np.zeros(len(y_arr))
    for tr_idx, val_idx in folds:
        m = LogisticRegression(C=C, max_iter=1000)
        m.fit(X_meta[tr_idx], y_arr[tr_idx])
        meta_oof[val_idx] = m.predict_proba(X_meta[val_idx])[:, 1]
    final_meta = LogisticRegression(C=C, max_iter=1000).fit(X_meta, y_arr)
    return meta_oof, roc_auc_score(y_arr, meta_oof), final_meta, names


ensemble_results = {}

equal_w = {n: 1.0 / len(oof_dict) for n in oof_dict}
ensemble_results["equal_avg"] = {"auc": roc_auc_score(y, blend_predictions(oof_dict, equal_w)),
                                  "weights": equal_w, "type": "prob"}

opt_w, opt_auc = optimize_weights_random_search(oof_dict, y)
ensemble_results["prob_blend_optimized"] = {"auc": opt_auc, "weights": opt_w, "type": "prob"}

oof_ranks = rank_transform(oof_dict)
rank_w, rank_auc = optimize_weights_random_search(oof_ranks, y)
ensemble_results["rank_blend_optimized"] = {"auc": rank_auc, "weights": rank_w, "type": "rank"}

meta_oof, meta_auc, final_meta, meta_names = stacking_cv(oof_dict, y)
ensemble_results["stacking"] = {"auc": meta_auc, "type": "stacking"}

print("=== Comparatif des strategies d'ensemble (AUC OOF) ===")
for k, v in sorted(ensemble_results.items(), key=lambda kv: -kv[1]["auc"]):
    print(f"  {k:24s} AUC = {v['auc']:.5f}")

best_key = max(ensemble_results, key=lambda k: ensemble_results[k]["auc"])
best_auc = ensemble_results[best_key]["auc"]
print(f"\n>>> Meilleure config: {best_key}  (OOF AUC = {best_auc:.5f})")

## 12. 🔍 Section critique : quel est le plafond de signal exploitable ?

Avant de conclure, vérifions si nos modèles **sous-exploitent** le signal disponible, ou s'ils
sont déjà proches du maximum atteignable. Méthode : un **target encoding OOF** (5-fold) de la
combinaison jointe des variables catégorielles les plus fortes + `Annual_Income_USD` binné en
quantiles (grossier, pour éviter la sur-fragmentation). Si ce "score naïf par groupe" est **déjà
inférieur** à l'AUC de nos GBDT, cela confirme que les modèles capturent déjà (quasi) tout le
signal disponible dans ces colonnes, et que le plafond observé (~0.94) est probablement un
**plancher de bruit intrinsèque au label**, pas une limite de nos modèles.

In [ ]:
KEY_STRONG = ["Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible",
              "Environmental_Concern_Level", "City_Type", "Current_Car_Type", "Gender"]

oracle_df = train.copy()
oracle_df["_y"] = y
oracle_df["income_q5"] = pd.qcut(oracle_df["Annual_Income_USD"], q=5, duplicates="drop").astype(str)
key_cols = KEY_STRONG + ["income_q5"]
global_mean = y.mean()

oof_oracle = np.full(len(oracle_df), np.nan)
for tr_idx, val_idx in FOLDS:
    tr = oracle_df.iloc[tr_idx]
    stats = tr.groupby(key_cols, observed=True)["_y"].agg(["mean", "count"]).reset_index()
    stats["smoothed"] = (stats["mean"] * stats["count"] + global_mean * 30.0) / (stats["count"] + 30.0)
    val_keys = oracle_df.iloc[val_idx][key_cols].reset_index(drop=True)
    merged = val_keys.merge(stats[key_cols + ["smoothed"]], on=key_cols, how="left")
    oof_oracle[val_idx] = merged["smoothed"].fillna(global_mean).to_numpy()

oracle_auc = roc_auc_score(y, oof_oracle)
print(f"AUC du score naif par groupe (categorielles fortes + revenu en 5 quantiles): {oracle_auc:.5f}")
print(f"AUC du meilleur GBDT individuel                                          : {summary_df['oof_auc'].max():.5f}")
print(f"AUC de l'ensemble final                                                  : {best_auc:.5f}")
print()
if best_auc > oracle_auc:
    print(">>> Les GBDT + l'ensemble depassent nettement le score naif par groupe:")
    print("    ils exploitent deja les variables continues bien au-dela d'un simple decoupage")
    print("    en quantiles grossiers. Le score plafonne donc probablement pour une raison de")
    print("    BRUIT INTRINSEQUE au label (le generateur du dataset a une composante aleatoire),")
    print("    et non par sous-exploitation du signal disponible.")

## 13. Génération de la soumission

Application de la meilleure configuration d'ensemble aux prédictions **test**, avec toutes
les vérifications Kaggle habituelles (forme, unicité des ids, absence de NaN, plage [0,1]).

In [ ]:
if ensemble_results[best_key]["type"] == "prob":
    test_pred_final = blend_predictions(test_dict, ensemble_results[best_key]["weights"])
elif ensemble_results[best_key]["type"] == "rank":
    test_pred_final = blend_predictions(rank_transform(test_dict), ensemble_results[best_key]["weights"])
else:  # stacking
    X_meta_test = np.column_stack([test_dict[n] for n in meta_names])
    test_pred_final = final_meta.predict_proba(X_meta_test)[:, 1]

submission = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), TARGET_COL: test_pred_final})

print("submission.shape       :", submission.shape)
print("id unique              :", submission[ID_COL].is_unique)
print("row count == test rows :", len(submission) == len(test))
print("NaN values              :", submission.isnull().sum().sum())
print("prediction range        : [{:.6f}, {:.6f}]".format(submission[TARGET_COL].min(), submission[TARGET_COL].max()))

assert submission[ID_COL].is_unique
assert len(submission) == len(test)
assert submission.isnull().sum().sum() == 0
assert submission[TARGET_COL].between(0, 1).all()
assert (test[ID_COL].to_numpy() == submission[ID_COL].to_numpy()).all()

submission.to_csv("submission.csv", index=False)
print("\nFichier 'submission.csv' ecrit avec succes.")
submission.head()

## 14. Conclusion

- 4 modèles de familles différentes ont été entraînés sur l'**intégralité** des 668 665 lignes
  d'entraînement, en validation croisée stratifiée à 5 plis, avec des hyperparamètres informés
  par une recherche Optuna dédiée.
- **Accélération GPU** : détection automatique de l'accélérateur (`nvidia-smi`) avec adaptation
  par modèle — CatBoost multi-GPU natif, LightGBM/XGBoost round-robin par pli — et repli CPU
  transparent en cas d'absence de GPU ou d'erreur, sans jamais faire échouer le notebook.
- L'**ensemble** (sélectionné automatiquement parmi blend pondéré / rank / stacking sur AUC OOF)
  constitue la meilleure estimation honnête de la performance en généralisation : voir la valeur
  de `best_auc` calculée en section 11 (typiquement **≈ 0.94 – 0.945** sur ce dataset).
- La section 12 démontre que ce plafond n'est pas un signe de sous-performance des modèles :
  même un oracle par groupement catégoriel simple plafonne en dessous de ce que nos GBDT
  atteignent déjà, ce qui pointe vers un **bruit intrinsèque au processus génératif du label**
  plutôt qu'un manque de feature engineering ou de puissance de modélisation.

### Pistes pour aller plus loin (si un score plus élevé est requis)
- **Plus de folds / seeds** : passer à 10-fold + moyenne multi-seed (gain marginal, +0.001~0.002).
- **Pseudo-labeling** : ré-entraîner en ajoutant les prédictions test les plus confiantes — risqué,
  à valider très soigneusement en CV pour éviter tout biais de confirmation.
- **Données externes** : si la compétition l'autorise, enrichir avec des données macro (prix de
  l'énergie, densité de bornes réelles par région, etc.) au-delà des colonnes fournies.
- **Deep learning tabulaire** (FT-Transformer, TabNet, embeddings d'entités) : rarement supérieur
  aux GBDT sur ce type de données tabulaires à faible cardinalité, mais apporte de la diversité
  supplémentaire à l'ensemble.